In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
print("Project root added to sys.path")

Project root added to sys.path


In [2]:
import torch
import torch.nn 
from torchvision import datasets, transforms
import numpy as np

from utils.data_partition import dirichlet_partition


torch.manual_seed(42)
np.random.seed(42)
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from clients.dense_client import FederatedDenseClient
from clients.aggregators import standard_fedavg_aggregate
from models.dense_models import get_dense_model
import flwr as fl
import numpy as np
import os
import pandas as pd

In [3]:

transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    
trainset = datasets.MNIST('./data', train=True, download=True, transform=transform)
testset = datasets.MNIST('./data', train=False, transform=transform)
    
    

print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")
print(f"Number of classes: {len(trainset.classes)}")

Training set size: 60000
Test set size: 10000
Number of classes: 10


In [4]:
num_clients = 5
partitions = dirichlet_partition(trainset, num_clients=num_clients, alpha=0.01)

c:\Users\la7tim\Desktop\Internship\FedTinyProp\utils\data_partition.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  labels = np.array(dataset.targets)


In [5]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import os
from datetime import datetime

from models.dense_models import get_dense_model
from clients.dense_client import FederatedDenseClient
from clients.aggregators import standard_fedavg_aggregate
from models.config import get_dense_config
from utils.save_results import save_training_logs_csv, append_to_training_log_csv

# Optional: set random seed for reproducibility
def set_seed(seed=42):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Global model evaluation
def evaluate_global_model(model, testset, device="cpu"):
    model.eval()
    model.to(device)
    test_loader = DataLoader(testset, batch_size=32)
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

def federated_training(partitions, testset, rounds=100, dataset_name="mnist", save_dir=None):
    # Create save directory with timestamp
    if save_dir is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_dir = f"results_{dataset_name}_{timestamp}"
    os.makedirs(save_dir, exist_ok=True)
    
    # Device setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Global model and config
    model = get_dense_model(dataset_name)
    config = get_dense_config(dataset_name)

    # Initialize metrics log with empty lists
    metrics_log = {
        "timestamp": [],
        "accuracy": [],
        "flops": [],
        "memory": [],
        "memory_saved": [],
        "communication": [],
        "sparsity": [],
        "skipped_batches": [],
        "effective_compute_ratio": [],
        "client_eval_history": [],
        "compression_ratio": [],
        "download_bytes": [],
        "upload_bytes": [],
        "model_size_bytes": []
    }

    # 2. Client initialization
    clients = []
    for i, dataset in enumerate(partitions):
        train_loader = DataLoader(dataset, batch_size=config["train"]["batch_size"], shuffle=True)
        test_loader = DataLoader(testset, batch_size=32)
        client_model = get_dense_model(dataset_name)
        client_model.load_state_dict(model.state_dict())
        clients.append(FederatedDenseClient(
            client_id=i,
            model=client_model,
            train_loader=train_loader,
            test_loader=test_loader,
            cfg=config,
            device=device
        ))

    # 3. Training loop
    for rnd in range(1, rounds + 1):
        current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\nRound {rnd} - {current_timestamp}")
        global_params = [val.cpu().numpy() for val in model.state_dict().values() if isinstance(val, torch.Tensor)]
        client_updates = []
        fit_metrics = []
        detailed_metrics = []

        # Client training phase
        for client in clients:
            # Get fit metrics
            params, _, fit_metric = client.fit(
                global_params,
                config={
                    "batch_size": config["train"]["batch_size"],
                    "local_epochs": config["train"]["local_epochs"]
                }
            )
            client_updates.append(params)
            fit_metrics.append(fit_metric)
            
            # Get detailed metrics
            detailed_metric = client.get_metrics()
            detailed_metrics.append(detailed_metric)

        # 4. Aggregate parameters
        model, agg_stats = standard_fedavg_aggregate(
            client_updates,
            model,
            dataset_sizes=[len(c.train_loader.dataset) for c in clients]
        )

        # 5. Evaluate global model
        test_accuracy = evaluate_global_model(model, testset, device=device)
        
        # 6. Calculate average metrics across clients
        # For dense baseline, we'll calculate communication metrics from client metrics
        model_size_bytes = sum(p.numel() * 4 for p in model.parameters())  # 4 bytes per float32
        upload_bytes = model_size_bytes * len(clients)  # ✅ only uploads
        download_bytes = model_size_bytes * len(clients)

        
        avg_metrics = {
            "timestamp": current_timestamp,
            "round": rnd,
            "accuracy": float(np.mean([m["accuracy"] for m in fit_metrics])),
            "test_accuracy": test_accuracy,
            "flops": float(np.mean([m["flops"] for m in detailed_metrics])),
            "memory": float(np.max([m["memory"] for m in detailed_metrics])),
            "memory_saved": float(np.mean([m.get("memory_saved", 0.0) for m in detailed_metrics])),
            "communication": float(download_bytes + upload_bytes),  # Total communication
            "sparsity": float(np.mean([m.get("sparsity", 0.0) for m in detailed_metrics])),
            "skipped_batches": int(np.sum([m.get("skipped_batches", 0) for m in detailed_metrics])),
            "effective_compute_ratio": float(np.mean([m.get("effective_compute_ratio", 1.0) for m in detailed_metrics])),
            "compression_ratio": float(np.mean([m.get("compression_ratio", 1.0) for m in detailed_metrics])),
            "download_bytes": float(download_bytes),
            "upload_bytes": float(upload_bytes),
            "model_size_bytes": float(model_size_bytes)
        }

        # Update metrics log
        # Update metrics log
        metrics_log["timestamp"].append(current_timestamp)  # Add this line
        for key in metrics_log:
            if key != "timestamp" and key != "client_eval_history":  # Modified condition
                if key in avg_metrics:
                        metrics_log[key].append(avg_metrics[key])
                else:
                    # Ensure all metrics arrays have the same length
                    metrics_log[key].append(0.0 if key != "client_eval_history" else {})
            elif key == "client_eval_history":
                metrics_log[key].append({})

        # 7. Save detailed metrics to CSV
        append_to_training_log_csv(
            os.path.join(save_dir, f"{dataset_name}_detailed_metrics.csv"),
            round_num=rnd,
            accuracy=avg_metrics["accuracy"],
            flops=avg_metrics["flops"],
            memory_bytes=avg_metrics["memory"],
            communication_bytes=avg_metrics["communication"],
            sparsity=avg_metrics["sparsity"],
            skipped_batches=avg_metrics["skipped_batches"],
            effective_compute_ratio=avg_metrics["effective_compute_ratio"],
            compression_ratio=avg_metrics["compression_ratio"],
            memory_saved=avg_metrics["memory_saved"],
            download_bytes=avg_metrics["download_bytes"],
            upload_bytes=avg_metrics["upload_bytes"],
            model_size_bytes=avg_metrics["model_size_bytes"]
        )

        # Save complete training logs periodically
        if rnd % 10 == 0 or rnd == rounds:
            save_training_logs_csv(
                os.path.join(save_dir, f"{dataset_name}_training_logs.csv"),
                metrics_log["accuracy"],
                metrics_log["flops"],
                metrics_log["memory"],
                metrics_log["communication"],
                metrics_log["sparsity"],
                memory_saved=metrics_log["memory_saved"],
                download_bytes=metrics_log["download_bytes"],
                upload_bytes=metrics_log["upload_bytes"],
                compression_ratio=metrics_log["compression_ratio"],
                model_size_bytes=metrics_log["model_size_bytes"]
            )

        print(f"\nRound {rnd} Metrics:")
        print(f"Accuracy: {avg_metrics['accuracy']:.2f}%")
        print(f"Test Accuracy: {test_accuracy:.2f}%")
        print(f"Communication: {avg_metrics['communication']/1024:.2f}KB")
        print(f"Compression Ratio: {avg_metrics['compression_ratio']:.2f}x")
        print(f"Download: {avg_metrics['download_bytes']/1024:.2f}KB")
        print(f"Upload: {avg_metrics['upload_bytes']/1024:.2f}KB")
        print(f"Model Size: {avg_metrics['model_size_bytes']/1024:.2f}KB")

    return model, metrics_log

# Example usage:
model, metrics = federated_training(partitions, testset, rounds=100, dataset_name="mnist")
# The metrics will be saved in the results directory with timestamp

Using device: cpu

Round 1 - 2025-05-19 14:36:17

[Server Debug] Starting dense aggregation...
[INFO] Appended round 1 metrics to results_mnist_20250519_143617\mnist_detailed_metrics.csv

Round 1 Metrics:
Accuracy: 89.49%
Test Accuracy: 39.53%
Communication: 63510.39KB
Compression Ratio: 1.00x
Download: 31755.20KB
Upload: 31755.20KB
Model Size: 6351.04KB

Round 2 - 2025-05-19 14:37:06

[Server Debug] Starting dense aggregation...
[INFO] Appended round 2 metrics to results_mnist_20250519_143617\mnist_detailed_metrics.csv

Round 2 Metrics:
Accuracy: 95.74%
Test Accuracy: 57.51%
Communication: 63510.39KB
Compression Ratio: 1.00x
Download: 31755.20KB
Upload: 31755.20KB
Model Size: 6351.04KB

Round 3 - 2025-05-19 14:37:54

[Server Debug] Starting dense aggregation...
[INFO] Appended round 3 metrics to results_mnist_20250519_143617\mnist_detailed_metrics.csv

Round 3 Metrics:
Accuracy: 96.58%
Test Accuracy: 59.72%
Communication: 63510.39KB
Compression Ratio: 1.00x
Download: 31755.20KB
Upload